# Mikado Judge — YOLO-OBB Training (Google Colab)

This notebook trains a YOLO-OBB model to detect Mikado sticks.  
**Runtime → Change runtime type → T4 GPU** before running.

## Workflow
1. Mount Google Drive and set paths
2. Install dependencies
3. Clone / pull the repository
4. Train the model
5. Evaluate and save weights to Drive

## 1. Setup — Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURE THESE to match your Google Drive layout
# ---------------------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/Data/mikado-judge'

# Path to the dataset archive you uploaded to Drive.
# Supports .zip (Windows Explorer) or .tar.gz (tar command).
# The archive should contain a single top-level folder called 'dataset'.
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.tar.gz'   # change extension if you used tar

# Where the dataset will be extracted (and where training reads from)
DATASET_DIR = f'{DRIVE_ROOT}/datasets/mikado'

# Where trained weights will be saved
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'

# ---------------------------------------------------------------------------
# Model selection
# ---------------------------------------------------------------------------
# yolo11s-obb:  9.7M params — fast, but struggles with thin 8px-wide sticks
# yolo11m-obb: 20.9M params — 3.3× more capacity, captures thin edge features
# yolo11l-obb: 25.3M params — diminishing returns vs m, needs batch=2
# On T4 (16GB): m-model fits comfortably at imgsz=1280, batch=2
BASE_MODEL = 'yolo11m-obb.pt'

# Training run name
RUN_NAME = 'mikado_v2'

# ---------------------------------------------------------------------------
# Training hyperparameters — tuned for thin elongated objects
# ---------------------------------------------------------------------------
EPOCHS = 200           # more epochs for 68-image dataset to converge
IMGSZ = 1280           # sticks ~8px wide (4624px originals)
BATCH = 2              # m-model needs ~10GB at 1280; batch=2 is safe on T4

import os
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Dataset archive: {DATASET_ZIP}')
print(f'Dataset dir:     {DATASET_DIR}')
print(f'Weights dir:     {WEIGHTS_DIR}')
print(f'Base model:      {BASE_MODEL}')
print(f'Image size:      {IMGSZ}')
print(f'Batch size:      {BATCH}')
print(f'Epochs:          {EPOCHS}')

In [ ]:
## Extract dataset archive (skips if already extracted)
import os, tarfile, zipfile
from pathlib import Path

def _extract(archive_path, dest_parent):
    """Extract .zip or .tar.gz archive into dest_parent."""
    p = str(archive_path)
    if p.endswith('.zip'):
        with zipfile.ZipFile(p, 'r') as f:
            f.extractall(dest_parent)
    else:  # .tar.gz / .tgz
        with tarfile.open(p, 'r:gz') as f:
            f.extractall(dest_parent)

if Path(DATASET_DIR).exists() and any(Path(DATASET_DIR).iterdir()):
    print(f'Dataset already extracted at {DATASET_DIR}, skipping.')
else:
    print(f'Extracting {DATASET_ZIP} ...')
    parent = str(Path(DATASET_DIR).parent)
    os.makedirs(parent, exist_ok=True)
    _extract(DATASET_ZIP, parent)
    # Archive root folder is called 'dataset'; rename to match DATASET_DIR
    extracted_root = os.path.join(parent, 'dataset')
    if os.path.exists(extracted_root) and extracted_root != DATASET_DIR:
        os.rename(extracted_root, DATASET_DIR)
    imgs = list(Path(DATASET_DIR).rglob('*.jpg')) + list(Path(DATASET_DIR).rglob('*.png'))
    print(f'Extracted to {DATASET_DIR} — {len(imgs)} images found.')

## 2. Install dependencies

In [ ]:
!pip install ultralytics --quiet
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Clone / update the repository

In [ ]:
REPO_DIR = '/content/mikado-judge'
REPO_URL = 'https://github.com/spyszkowski/mikado-judge.git'  # Update this!

import os
if os.path.exists(REPO_DIR):
    print('Repository exists, pulling latest changes...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}

# Install as editable package so imports work
!pip install -e {REPO_DIR} --quiet
print('Package installed.')

## 4. Verify dataset

In [ ]:
import os
from pathlib import Path

dataset_path = Path(DATASET_DIR)
train_images = list((dataset_path / 'images' / 'train').glob('*'))
val_images = list((dataset_path / 'images' / 'val').glob('*'))
train_labels = list((dataset_path / 'labels' / 'train').glob('*.txt'))
val_labels = list((dataset_path / 'labels' / 'val').glob('*.txt'))

print(f'Train images: {len(train_images)}')
print(f'Train labels: {len(train_labels)}')
print(f'Val images:   {len(val_images)}')
print(f'Val labels:   {len(val_labels)}')

if len(train_images) == 0:
    print('\n⚠ WARNING: No training images found. Upload your dataset to Drive first.')
else:
    print('\n✓ Dataset looks good.')

In [ ]:
# Write a dataset.yaml pointing to the Colab paths
import yaml

dataset_yaml_path = f'{DATASET_DIR}/dataset.yaml'
dataset_yaml = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 5,
    'names': {0: 'mikado', 1: 'blue', 2: 'red', 3: 'yellow', 4: 'green'},
}
with open(dataset_yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)
print(f'Written: {dataset_yaml_path}')

## 5. Train the model

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=dataset_yaml_path,
    task='obb',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=RUN_NAME,
    project='/content/runs/obb',
    exist_ok=True,

    # === Loss weights — optimized for thin elongated objects ===
    # Default YOLO: box=7.5, cls=0.5, dfl=1.5
    # Thin sticks need precise localization (high box) and less
    # classification pressure (tips are tiny vs full stick length).
    box=10.0,            # +33% — prioritize precise OBB localization
    cls=0.3,             # -40% — classify later, detect first
    dfl=2.0,             # +33% — sharper box edges for thin objects

    # === Optimizer — conservative for 68-image dataset ===
    optimizer='AdamW',   # better convergence on small datasets than SGD
    lr0=0.001,           # 10× lower than YOLO default (0.01)
    lrf=0.1,             # final LR = lr0 × 0.1 = 0.0001
    cos_lr=True,         # cosine schedule — smooth convergence
    warmup_epochs=5,     # longer warmup prevents early divergence
    weight_decay=0.001,  # regularization against overfit

    # === Early stopping ===
    patience=30,         # stop if no improvement for 30 epochs
    save_period=10,      # checkpoint every 10 epochs for recovery

    # === Augmentation — reduced aggressiveness for thin objects ===
    # Thin sticks (8px wide at 1280) are fragile under heavy augmentation.
    # Mosaic cuts sticks at boundaries; heavy scale can make them invisible.
    degrees=90,          # rotation — sticks are at all angles already
    mosaic=0.5,          # reduced — full mosaic cuts thin sticks at seams
    close_mosaic=20,     # disable mosaic for last 20 epochs (fine-tuning)
    copy_paste=0.3,      # good for overlapping objects
    mixup=0.0,           # DISABLED — overlapping stick piles = nonsense
    erasing=0.1,         # mild — heavy erasing deletes thin sticks entirely
    shear=0.0,           # DISABLED — distorts OBB angles
    scale=0.3,           # reduced from 0.5 — at 0.5 sticks can become 4px (invisible)
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    fliplr=0.5,
    flipud=0.5,

    # === Advanced ===
    amp=True,            # mixed precision — faster + less VRAM
    dropout=0.1,         # regularization for tiny dataset
    cache='ram',
    pretrained=True,
    verbose=True,
)

## 5b. Resume interrupted training (run this instead of cell 5 if training was interrupted)

In [ ]:
# Uncomment and set the path to resume training
# RESUME_WEIGHTS = f'/content/runs/obb/{RUN_NAME}/weights/last.pt'
# from ultralytics import YOLO
# model = YOLO(RESUME_WEIGHTS)
# model.train(resume=True)

## 6. Evaluate the trained model

In [ ]:
best_weights = f'/content/runs/obb/{RUN_NAME}/weights/best.pt'
model = YOLO(best_weights)
metrics = model.val(data=dataset_yaml_path, task='obb', imgsz=IMGSZ)
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 7. Visualise sample predictions

In [ ]:
import random
from pathlib import Path
from IPython.display import display
from PIL import Image

best_weights = f'/content/runs/obb/{RUN_NAME}/weights/best.pt'
model = YOLO(best_weights)

val_imgs = list((Path(DATASET_DIR) / 'images' / 'val').glob('*'))
samples = random.sample(val_imgs, min(4, len(val_imgs)))

for img_path in samples:
    result = model.predict(str(img_path), imgsz=IMGSZ, conf=0.1, verbose=False)[0]
    vis = result.plot(line_width=4)   # thicker lines — sticks are thin
    # Downscale for display (original images are ~4600px wide)
    img = Image.fromarray(vis[:, :, ::-1])
    w, h = img.size
    img = img.resize((min(w, 1200), int(h * min(w, 1200) / w)), Image.LANCZOS)
    n = len(result.obb) if result.obb is not None else 0
    print(f'{img_path.name}: {n} detections')
    display(img)

## 8. Save best weights to Google Drive

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{WEIGHTS_DIR}/{RUN_NAME}_{timestamp}_best.pt'
shutil.copy2(best_weights, dest)
print(f'Saved weights → {dest}')

# Also copy training curves
results_csv = f'/content/runs/obb/{RUN_NAME}/results.csv'
if os.path.exists(results_csv):
    shutil.copy2(results_csv, f'{WEIGHTS_DIR}/{RUN_NAME}_{timestamp}_results.csv')
    print('Saved training results CSV.')